# 第 06 章：基础设施 (Observability & Persistence)

## 3. 场景演练 B：持久化读档 (MemorySaver)

### 💡 核心概念：什么是 Config？

在 LangChain/LangGraph 中，调用 Agent 时除了输入 `input`，通常还会传入一个 `config` 字典。它的结构通常如下：

```python
config = {
    "configurable": {
        "thread_id": "user_001"  # 👈 这是对话的“卡槽 ID”
    }
}
```

**为什么要这么设计？**
1. **侧信道传输**：`thread_id` 是给系统架构看的（用于找回数据库里的存档），不应该混合在给 AI 看的对话内容里。
2. **多租户隔离**：通过更换 `thread_id`，同一个 Agent 实例可以同时为成千上万个用户提供独立的、互不干扰的对话空间。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://api.deepseek.com"
)

# 1. 初始化持久化层（硬盘）
checkpointer = MemorySaver()

# 2. 绑定 Checkpointer
# 注意：我们不传 tools，让它作为一个纯聊天 Agent 演示记忆
agent_executor = create_react_agent(llm, tools=[], checkpointer=checkpointer)

print("带有‘记忆读写头’的 Agent 已就绪。")

### 🧪 实验 1：建立存档 (thread_id: 'alice')
我们告诉 Agent 我们的爱好。注意：我们避开了敏感的“密码”字样，改为普通生活信息。

In [ ]:
# 定义 Alice 的专属卡槽
alice_config = {"configurable": {"thread_id": "alice_session_001"}}

print("--- Alice 发起对话 ---")
input_1 = {"messages": [HumanMessage(content="你好，我叫 Alice，我超级喜欢喝冰美式咖啡。")]}
res1 = agent_executor.invoke(input_1, config=alice_config)

print(f"Agent 对 Alice 的回复: {res1['messages'][-1].content}")

### 🧪 实验 2：跨调用读档
只要我们传入相同的 `alice_config`，Agent 就能从 MemorySaver 中找回之前的对话状态。

In [ ]:
print("--- Alice 再次提问 ---")
input_2 = {"messages": [HumanMessage(content="还记得我喜欢喝什么吗？")]}
res2 = agent_executor.invoke(input_2, config=alice_config)

print(f"Agent 的记忆回复: {res2['messages'][-1].content}")

### 🧪 实验 3：ID 隔离验证 (thread_id: 'bob')
如果我们换成 Bob 的卡槽，Agent 应该完全不认识 Alice。

In [ ]:
bob_config = {"configurable": {"thread_id": "bob_session_999"}}

print("--- Bob 乱入对话 ---")
input_3 = {"messages": [HumanMessage(content="你好，你知道我喜欢喝什么吗？")]}
res3 = agent_executor.invoke(input_3, config=bob_config)

print(f"Agent 对 Bob 的回复: {res3['messages'][-1].content}")

### 🕵️‍♂️ 深度解剖：Config 到底对 Agent 做了什么？
让我们直接打印出当前的持久化数据，看看 thread_id 对应的快照内容。

In [ ]:
# 获取 Alice 线程的最新存档
checkpoint = checkpointer.get(alice_config)

print("--- Alice 的大脑快照数据 ---")
print(f"存储位置 ID: {checkpoint['metadata']['thread_id']}")
print(f"对话轮次: {len(checkpoint['channel_values']['messages'])}")
print(f"最后一条记忆: {checkpoint['channel_values']['messages'][-1].content[:30]}...")